# Proyecto 2 Bioseñales 

**Estudiantes:**

Luisa María Hernández Quintero 

Karen Agudelo Toro 

In [1]:
import mne
import matplotlib.pyplot as plt
import numpy as np
from tabulate import tabulate
import pandas as pd
import os
import mne
import os
import glob
import numpy as np
import pandas as pd
from tabulate import tabulate
from mne.decoding import CSP
from scipy.signal import coherence
from antropy import perm_entropy


## Punto 1: Función de procesamiento de señales y de índices  

### Función de procesamiento de señal 

In [2]:
#Definicion de funciones necesarias para el procesamiento
def procesar_archivo_unificada(ruta):
    #Extraccion del sujeto y run 
    nombre = os.path.basename(ruta)
    partes = nombre.split('_')
    
    # Extraccion del sujeto
    sujeto = partes[0]
    #Extraccion del run
    run_con_extension = partes[2].split('-')[1]
    run = run_con_extension.split('.')[0]
    
    # Carga del EEG con MNE
    raw = mne.io.read_raw_eeglab(ruta, preload=True, verbose=False)
    
    # Se aplica el filtro Notch a 60Hz y el pasa-banda genérico de 0.5 a 45 Hz
    raw.notch_filter(60, verbose=False)
    raw.filter(0.5, 45, verbose=False)
    
    # Extraccion de eventos 
    events, event_id = mne.events_from_annotations(raw, verbose=False)
    #Mapeo de eventos
    eventos_dict = {
        'reposo':    event_id.get('TASK1T0') or event_id.get('TASK2T0'),
        'izquierda': event_id.get('TASK1T1') or event_id.get('TASK2T1'),
        'derecha':   event_id.get('TASK1T2') or event_id.get('TASK2T2')
    }
    
    # Se limpian los eventos inexistentes
    eventos_dict = {k: v for k, v in eventos_dict.items() if v is not None}
    
    if not eventos_dict:
        print(f"Advertencia: No se encontraron eventos esperados en {nombre}")
        return None, None, None, None, None, None, None, None, None, sujeto + '_' + run, run

    # Se define la lista de canales garantizados en la corteza motora para el CSP
    canales_motores_csp = ['C5', 'C3', 'C1', 'Cz', 'C2', 'C4', 'C6']
    
    #ventanas temporales
    tmin, tmax = 0.0, 2.0
    
  
    # GENERACIÓN DEL BLOQUE 0: Para PSD y Coherencia Espectral
  
    raw_bloque0 = raw.copy().pick(['C3', 'Cz', 'C4'])
    epocas_b0 = mne.Epochs(raw_bloque0, events, event_id=eventos_dict, tmin=tmin, tmax=tmax, baseline=None, preload=True, verbose=False)
    
    #separacion por condicion
    b0_er = epocas_b0['reposo'] if 'reposo' in epocas_b0.event_id else None
    b0_ei = epocas_b0['izquierda'] if 'izquierda' in epocas_b0.event_id else None
    b0_ed = epocas_b0['derecha'] if 'derecha' in epocas_b0.event_id else None
    
   
    # GENERACIÓN DEL BLOQUE 1: Para Patrones Espaciales Comunes (CSP)
    raw_bloque1 = raw.copy().pick(canales_motores_csp)
    epocas_b1 = mne.Epochs(raw_bloque1, events, event_id=eventos_dict, tmin=tmin, tmax=tmax, baseline=None, preload=True, verbose=False)
    
    ep_rep_csp = epocas_b1['reposo'] if 'reposo' in epocas_b1.event_id else None
    ep_izq_csp = epocas_b1['izquierda'] if 'izquierda' in epocas_b1.event_id else None
    ep_der_csp = epocas_b1['derecha'] if 'derecha' in epocas_b1.event_id else None
    
    #filtrado especifico para bandas mu y beta
    b1_er = {'mu': ep_rep_csp.copy().filter(8.0, 13.0, verbose=False), 'beta': ep_rep_csp.copy().filter(13.0, 30.0, verbose=False)} if ep_rep_csp is not None else None
    b1_ei = {'mu': ep_izq_csp.copy().filter(8.0, 13.0, verbose=False), 'beta': ep_izq_csp.copy().filter(13.0, 30.0, verbose=False)} if ep_izq_csp is not None else None
    b1_ed = {'mu': ep_der_csp.copy().filter(8.0, 13.0, verbose=False), 'beta': ep_der_csp.copy().filter(13.0, 30.0, verbose=False)} if ep_der_csp is not None else None
    
   
    # GENERACIÓN DEL BLOQUE 2: Para Entropía de Permutación
    b2_er = {'mu': b0_er.copy().filter(8.0, 13.0, verbose=False), 'beta': b0_er.copy().filter(13.0, 30.0, verbose=False)} if b0_er is not None else None
    b2_ei = {'mu': b0_ei.copy().filter(8.0, 13.0, verbose=False), 'beta': b0_ei.copy().filter(13.0, 30.0, verbose=False)} if b0_ei is not None else None
    b2_ed = {'mu': b0_ed.copy().filter(8.0, 13.0, verbose=False), 'beta': b0_ed.copy().filter(13.0, 30.0, verbose=False)} if b0_ed is not None else None
    
    return (b0_er, b0_ei, b0_ed, b1_er, b1_ei, b1_ed, b2_er, b2_ei, b2_ed, sujeto, run)

In [3]:
# 1. Ejecutar la función para extraer todo
b0_er, b0_ei, b0_ed, b1_er, b1_ei, b1_ed, b2_er, b2_ei, b2_ed, s, r = procesar_archivo_unificada('sub-001_task-motion_run-4_eeg.set')

# 2. Visualizacion de resultados
print(f"SUJETO: {s} | RUN: {r}\n")

print("--- BLOQUE 0: Para Potencia (PSD) y Coherencia ---")
print(f"Reposo (3 canales: C3, Cz, C4)    : {b0_er}")
print(f"Izquierda (3 canales: C3, Cz, C4) : {b0_ei}")
print(f"Derecha (3 canales: C3, Cz, C4)   : {b0_ed}\n")

print("--- BLOQUE 1: Para CSP (7 canales para geometría) ---")
print(f"Izquierda en Ritmo Mu (8-13 Hz)   : {b1_ei['mu'] if b1_ei else None}")
print(f"Izquierda en Ritmo Beta (13-30 Hz): {b1_ei['beta'] if b1_ei else None}")
print(f"Derecha en Ritmo Mu (8-13 Hz)     : {b1_ed['mu'] if b1_ed else None}")
print(f"Derecha en Ritmo Beta (13-30 Hz)  : {b1_ed['beta'] if b1_ed else None}\n")

print("--- BLOQUE 2: Para Entropía (3 canales filtrados) ---")
print(f"Reposo en Ritmo Mu (8-13 Hz)      : {b2_er['mu'] if b2_er else None}")
print(f"Izquierda en Ritmo Beta (13-30 Hz): {b2_ei['beta'] if b2_ei else None}")
print(f"Derecha en Ritmo Beta (13-30 Hz)  : {b2_ed['beta'] if b2_ed else None}")

SUJETO: sub-001 | RUN: 4

--- BLOQUE 0: Para Potencia (PSD) y Coherencia ---
Reposo (3 canales: C3, Cz, C4)    : <Epochs | 15 events (all good), 0 – 2 s (baseline off), ~144 KiB, data loaded,
 'reposo': 15>
Izquierda (3 canales: C3, Cz, C4) : <Epochs | 8 events (all good), 0 – 2 s (baseline off), ~91 KiB, data loaded,
 'izquierda': 8>
Derecha (3 canales: C3, Cz, C4)   : <Epochs | 7 events (all good), 0 – 2 s (baseline off), ~84 KiB, data loaded,
 'derecha': 7>

--- BLOQUE 1: Para CSP (7 canales para geometría) ---
Izquierda en Ritmo Mu (8-13 Hz)   : <Epochs | 8 events (all good), 0 – 2 s (baseline off), ~174 KiB, data loaded,
 'izquierda': 8>
Izquierda en Ritmo Beta (13-30 Hz): <Epochs | 8 events (all good), 0 – 2 s (baseline off), ~174 KiB, data loaded,
 'izquierda': 8>
Derecha en Ritmo Mu (8-13 Hz)     : <Epochs | 7 events (all good), 0 – 2 s (baseline off), ~157 KiB, data loaded,
 'derecha': 7>
Derecha en Ritmo Beta (13-30 Hz)  : <Epochs | 7 events (all good), 0 – 2 s (baseline off)

### Función de cálculo de PSD 

In [ ]:
def calcular_psd_epochs(epocas):
    """
    Calcula la Densidad Espectral de Potencia (PSD) usando el método de Welch
    para las épocas de los canales seleccionados (C3, Cz, C4).
    """
    # Verificación de seguridad por si el archivo fue descartado previamente
    if epocas is None:
        return None, None
    
    #calculo de PSD con el metodo de WELCH    
    psds_obj = epocas.compute_psd(
        method='welch',
        fmin=0.5,       # Alineado con el filtro pasa-altas  para derivadas lentas y artefactos DC
        fmax=45.0,      # Alineado con el filtro pasa-bajas, evita frecuencias musculares altas y ruido
        n_fft=256,      #numero de puntos para la FFT
        n_overlap=128,  #numero de puntos de solapamiento
        verbose=False
    )
    
    # Extraemos los datos (epochs, canales, frecuencias) y el vector de frecuencias
    return psds_obj.get_data(), psds_obj.freqs


def potencia_banda(psds, freqs, fmin, fmax):
    """
    Calcula la potencia promedio en un rango de frecuencias específico (fmin a fmax)
    para cada época y cada canal.
    """
    if psds is None or freqs is None:
        return None

    idx = (freqs >= fmin) & (freqs <= fmax)
    
    # psds tiene la forma: (n_epocas, n_canales, n_frecuencias)
    # Calculamos el promedio en el eje 2 (frecuencias)
    potencia = np.mean(psds[:, :, idx], axis=2)
    
    # Retorna una matriz de forma (n_epocas, n_canales) -> (Épocas, 3 canales)
    return potencia

### Función de cálculo de MSC

In [24]:
def calcular_msc_epochs(epocas, fs=160):
    """
    Calcula la coherencia espectral por época
    entre pares de canales.
    
    Retorna:
    coherencias -> diccionario con matrices
    freqs -> vector de frecuencias
    """
    
    if epocas is None:
        return None, None
    
    datos = epocas.get_data() #obtenemos: n_epocas, n_canales, n_muestras
    
    pares = {
        'C3_C4': (0, 2),
        'C3_Cz': (0, 1),
        'C4_Cz': (2, 1)
    }
    
    resultados = {}
    
    for nombre_par, (ch1, ch2) in pares.items():
        
        coherencias_par = []
        
        for epoch in datos:
            
            señal1 = epoch[ch1]
            señal2 = epoch[ch2]
            
            freqs, coh = coherence(
                señal1,
                señal2,
                fs=fs,
                nperseg=128
            )
            
            coherencias_par.append(coh)
        
        resultados[nombre_par] = np.array(coherencias_par)
    
    return resultados, freqs

#Extraer coherencia por banda
def coherencia_banda(coherencias, freqs, fmin, fmax):
    """
    Calcula la coherencia promedio
    en una banda de frecuencia.
    """
    
    idx = (freqs >= fmin) & (freqs <= fmax)
    
    resultados = {}
    
    for par, valores in coherencias.items():
        
        resultados[par] = np.mean(
            valores[:, idx],
            axis=1
        )
    
    return resultados
#Cada vector contiene una coherencia promedio por epoca



### Función de cálculo de entropia 

In [ ]:
#Funcion entropia
def calcular_entropia_epochs(epocas, orden=3, delay=1):
    """
    Calcula entropía de permutación
    por época y por canal.
    """
    
    if epocas is None:
        return None
    
    datos = epocas.get_data()
    
    resultados = []
    
    for epoch in datos:
        
        entropias_canales = []
        
        for canal in epoch:
            
            pe = perm_entropy(
                canal,
                order=orden,
                delay=delay,
                normalize=True
            )
            
            entropias_canales.append(pe)
        
        resultados.append(entropias_canales)
    
    return np.array(resultados)
#retorna una matriz: (n_epocas, n_canales) con la entropía de cada canal por época

## Punto 2 

### Construcción de data frame principal 

In [7]:
#Se hace una función aparte para el dataframe de PSD porque PSD necesita 2 niveles, primero se obtiene toda la curva espectral y de alli, se selccionan bandas, etc.
def construir_dataframe_psd(ep_reposo, ep_izq, ep_der, sujeto, run):

    filas = []

    tareas_mapeo = {
        'reposo': (ep_reposo, 0),
        'izquierda': (ep_izq, 1),
        'derecha': (ep_der, 2)
    }

    for nombre_tarea, (objeto_epocas, codigo_tarea) in tareas_mapeo.items():

        if objeto_epocas is None:
            continue

        psds, freqs = calcular_psd_epochs(objeto_epocas)

        potencia_mu = potencia_banda(psds, freqs, 8, 13)
        potencia_beta = potencia_banda(psds, freqs, 13, 30)

        n_epocas = potencia_mu.shape[0]

        for i in range(n_epocas):

            fila = {
                'Sujeto': sujeto,
                'run': run,
                'tarea': codigo_tarea,
                'epoch_id': i,

                'PSD_mu_C3': potencia_mu[i, 0],
                'PSD_mu_Cz': potencia_mu[i, 1],
                'PSD_mu_C4': potencia_mu[i, 2],

                'PSD_beta_C3': potencia_beta[i, 0],
                'PSD_beta_Cz': potencia_beta[i, 1],
                'PSD_beta_C4': potencia_beta[i, 2]
            }

            filas.append(fila)

    return pd.DataFrame(filas)

### Extracción para incluir en data frame índices consultados 

#### Extracción de datos de CSP

In [10]:

def extraer_features_csp(b1_er, b1_ei, b1_ed, sujeto, run, ritmo='mu'):
    """
    Extrae características CSP entrenando con Izquierda y Derecha,
    y proyecta también el Reposo (Tarea 0). Construye un DataFrame unificado.
    """

    if b1_er is None or b1_ei is None or b1_ed is None:
        print(f"Faltan épocas para CSP en {sujeto} run {run}.")
        return None, None

    # 1. EXTRACCIÓN DE SEÑALES EN EL TIEMPO
    X_rep = b1_er[ritmo].get_data()  # Datos Reposo
    X_izq = b1_ei[ritmo].get_data()  # Datos Izquierda
    X_der = b1_ed[ritmo].get_data()  # Datos Derecha

    # Etiquetas numéricas para las tareas activas
    y_izq = np.ones(X_izq.shape[0]) * 1
    y_der = np.ones(X_der.shape[0]) * 2

    # 2. CONCATENACIÓN ESTRICTA PARA ENTRENAMIENTO (Solo clases activas)
    X_tren = np.concatenate((X_izq, X_der), axis=0)
    y_tren = np.concatenate((y_izq, y_der), axis=0)

    # 3. CONFIGURACIÓN DEL MODELO CSP
    csp_modelo = CSP(
        n_components=4,
        reg=None,
        log=True,
        transform_into='average_power'
    )

    # El modelo se entrena y se aplica a los datos de movimiento
    features_movimiento = csp_modelo.fit_transform(X_tren, y_tren)
    
    # 4. EL TRUCO DE INGENIERÍA: Proyectar el reposo usando el filtro entrenado
    # Aplicamos las mismas ecuaciones espaciales al bloque de calma cerebral
    features_reposo = csp_modelo.transform(X_rep)

    # 5. CONSTRUCCIÓN DE LAS COLUMNAS DE CARACTERÍSTICAS
    columnas = [
        f'CSP_{ritmo}_comp1',
        f'CSP_{ritmo}_comp2',
        f'CSP_{ritmo}_comp3',
        f'CSP_{ritmo}_comp4'
    ]

    # Unimos los renglones matemáticos de las 3 tareas en una sola matriz gigante
    features_total = np.concatenate((features_reposo, features_movimiento), axis=0)
    df_csp = pd.DataFrame(features_total, columns=columnas)

    # 6. ASIGNACIÓN ORDENADA DE VARIABLES DE CONTROL
    # Generamos el vector de etiquetas final respetando el nuevo orden exacto
    y_total = np.concatenate((
        np.zeros(X_rep.shape[0]),  # Tarea 0 (Reposo)
        y_izq,                     # Tarea 1 (Izquierda)
        y_der                      # Tarea 2 (Derecha)
    ), axis=0)
    
    df_csp['tarea'] = y_total.astype(int)
    df_csp['Sujeto'] = sujeto
    df_csp['run'] = run

    # Separación y conteo de ID de épocas por cada clase independiente
    n_rep = X_rep.shape[0]
    n_izq = X_izq.shape[0]
    n_der = X_der.shape[0]

    epoch_rep = list(range(n_rep))
    epoch_izq = list(range(n_izq))
    epoch_der = list(range(n_der))

    df_csp['epoch_id'] = epoch_rep + epoch_izq + epoch_der

    # Reordenamiento estético de las columnas para el dataset maestro
    columnas_finales = [
        'Sujeto',
        'run',
        'tarea',
        'epoch_id'
    ] + columnas

    df_csp = df_csp[columnas_finales]

    return df_csp, csp_modelo

comp1 (El especialista en la Izquierda): Es un filtro diseñado para amplificar al máximo la señal cuando imaginas mover la mano izquierda y apagarla por completo cuando imaginas la derecha. 

comp2 (El asistente de la Izquierda): Hace un trabajo similar al primero, pero captura detalles secundarios o sutiles del hemisferio derecho del cerebro que ayudan a confirmar la tarea.

comp3 (El asistente de la Derecha): Es el gemelo del componente 2, pero empieza a buscar patrones que favorecen la detección de la mano derecha.

comp4 (El especialista en la Derecha): Es el filtro diseñado para hacer lo contrario al primero: amplifica la señal al máximo cuando imaginas mover la mano derecha y la destruye por completo si piensas en la izquierda.

#### Extracción de datos de MSC

In [11]:
def extraer_features_msc(b0_er, b0_ei, b0_ed, sujeto, run, ritmo='mu'):
    """
    Extrae características de coherencia espectral (MSC) para 
    Reposo (0), Izquierda (1) y Derecha (2) en un DataFrame unificado.
    """
    if b0_er is None or b0_ei is None or b0_ed is None:
        print(f"Faltan épocas para calcular coherencia en {sujeto} run {run}.")
        return None
    
    # Calcular coherencias usando tus funciones matemáticas originales
    coh_r, freqs = calcular_msc_epochs(b0_er)
    coh_i, _     = calcular_msc_epochs(b0_ei)
    coh_d, _     = calcular_msc_epochs(b0_ed)
    
    fmin, fmax = (8, 13) if ritmo == 'mu' else (13, 30)
    
    # Filtrar por la banda elegida
    coh_b_rep = coherencia_banda(coh_r, freqs, fmin, fmax)
    coh_b_izq = coherencia_banda(coh_i, freqs, fmin, fmax)
    coh_b_der = coherencia_banda(coh_d, freqs, fmin, fmax)
    
    filas = []
    
    # --- TAREA 0: REPOSO ---
    for i in range(len(coh_b_rep['C3_C4'])):
        filas.append({
            'Sujeto': sujeto, 'run': run, 'tarea': 0, 'epoch_id': i,
            f'MSC_{ritmo}_C3_C4': coh_b_rep['C3_C4'][i],
            f'MSC_{ritmo}_C3_Cz': coh_b_rep['C3_Cz'][i],
            f'MSC_{ritmo}_C4_Cz': coh_b_rep['C4_Cz'][i]
        })
        
    # --- TAREA 1: MANO IZQUIERDA ---
    for i in range(len(coh_b_izq['C3_C4'])):
        filas.append({
            'Sujeto': sujeto, 'run': run, 'tarea': 1, 'epoch_id': i,
            f'MSC_{ritmo}_C3_C4': coh_b_izq['C3_C4'][i],
            f'MSC_{ritmo}_C3_Cz': coh_b_izq['C3_Cz'][i],
            f'MSC_{ritmo}_C4_Cz': coh_b_izq['C4_Cz'][i]
        })
    
    # --- TAREA 2: MANO DERECHA ---
    for i in range(len(coh_b_der['C3_C4'])):
        filas.append({
            'Sujeto': sujeto, 'run': run, 'tarea': 2, 'epoch_id': i,
            f'MSC_{ritmo}_C3_C4': coh_b_der['C3_C4'][i],
            f'MSC_{ritmo}_C3_Cz': coh_b_der['C3_Cz'][i],
            f'MSC_{ritmo}_C4_Cz': coh_b_der['C4_Cz'][i]
        })
    
    return pd.DataFrame(filas)

#### Extracción de datos de entropia 

In [12]:
def extraer_features_entropia(b2_er, b2_ei, b2_ed, sujeto, run, ritmo='mu'):
    """
    Extrae características de Entropía de Permutación para
    Reposo (0), Izquierda (1) y Derecha (2) en un DataFrame unificado.
    """
    if b2_er is None or b2_ei is None or b2_ed is None:
        print(f"Faltan épocas para calcular entropía en {sujeto} run {run}.")
        return None
    
    # Calcular entropías usando tu función matemática original
    pe_rep = calcular_entropia_epochs(b2_er)
    pe_izq = calcular_entropia_epochs(b2_ei)
    pe_der = calcular_entropia_epochs(b2_ed)
    
    filas = []
    
    # --- TAREA 0: REPOSO ---
    for i, epoch in enumerate(pe_rep):
        filas.append({
            'Sujeto': sujeto, 'run': run, 'tarea': 0, 'epoch_id': i,
            f'PE_{ritmo}_C3': epoch[0],
            f'PE_{ritmo}_Cz': epoch[1],
            f'PE_{ritmo}_C4': epoch[2]
        })
        
    # --- TAREA 1: MANO IZQUIERDA ---
    for i, epoch in enumerate(pe_izq):
        filas.append({
            'Sujeto': sujeto, 'run': run, 'tarea': 1, 'epoch_id': i,
            f'PE_{ritmo}_C3': epoch[0],
            f'PE_{ritmo}_Cz': epoch[1],
            f'PE_{ritmo}_C4': epoch[2]
        })
        
    # --- TAREA 2: MANO DERECHA ---
    for i, epoch in enumerate(pe_der):
        filas.append({
            'Sujeto': sujeto, 'run': run, 'tarea': 2, 'epoch_id': i,
            f'PE_{ritmo}_C3': epoch[0],
            f'PE_{ritmo}_Cz': epoch[1],
            f'PE_{ritmo}_C4': epoch[2]
        })
        
    return pd.DataFrame(filas)

### Procesamiento de un solo archivo 

In [ ]:
# 1. PROCESAMIENTO UNIFICADO DEL ARCHIVO ORIGINAL
(
    b0_er, b0_ei, b0_ed,
    b1_er, b1_ei, b1_ed,
    b2_er, b2_ei, b2_ed,
    sujeto,
    run
) = procesar_archivo_unificada('sub-001_task-motion_run-4_eeg.set')

# Claves de cruce idénticas para asegurar la alineación horizontal perfecta
claves_cruce = ['Sujeto', 'run', 'tarea', 'epoch_id']


# 2. EXTRACCIÓN DE CARACTERÍSTICAS (Asegurando las 3 tareas en cada índice)

# DENSIDAD ESPECTRAL DE POTENCIA (PSD) 
df_psd = construir_dataframe_psd(b0_er, b0_ei, b0_ed, sujeto, run)

# PATRONES ESPACIALES COMUNES (CSP) 
df_csp_mu, modelo_mu     = extraer_features_csp(b1_er, b1_ei, b1_ed, sujeto, run, ritmo='mu')
df_csp_beta, modelo_beta = extraer_features_csp(b1_er, b1_ei, b1_ed, sujeto, run, ritmo='beta')

# COHERENCIA ESPECTRAL (MSC) 
df_msc_mu   = extraer_features_msc(b0_er, b0_ei, b0_ed, sujeto, run, ritmo='mu')
df_msc_beta = extraer_features_msc(b0_er, b0_ei, b0_ed, sujeto, run, ritmo='beta')

# ENTROPÍA DE PERMUTACIÓN (PE)
df_pe_mu   = extraer_features_entropia(b2_er['mu'], b2_ei['mu'], b2_ed['mu'], sujeto, run, ritmo='mu')
df_pe_beta = extraer_features_entropia(b2_er['beta'], b2_ei['beta'], b2_ed['beta'], sujeto, run, ritmo='beta')


# 3. FUSIÓN HORIZONTAL ESTRICTA
# Inicializamos el dataframe unificado usando la potencia como base
df_total = df_psd.copy()

# Acoplamos cada descriptor uno por uno garantizando consistencia simétrica
df_total = df_total.merge(df_csp_mu,   on=claves_cruce, how='inner')
df_total = df_total.merge(df_csp_beta, on=claves_cruce, how='inner')
df_total = df_total.merge(df_msc_mu,   on=claves_cruce, how='inner')
df_total = df_total.merge(df_msc_beta, on=claves_cruce, how='inner')
df_total = df_total.merge(df_pe_mu,   on=claves_cruce, how='inner')
df_total = df_total.merge(df_pe_beta,  on=claves_cruce, how='inner')



In [ ]:

# 1. Configuración absoluta de visualización 
pd.set_option('display.max_columns', None)  # Fuerza a mostrar todas las columnas
pd.set_option('display.max_rows', None)    
pd.set_option('display.width', 1000)        

# 2. Imprimir las dimensiones reales del archivo procesado
print(f"DIMENSIONES DE ESTE ARCHIVO: {df_total.shape} (Filas, Columnas)\n")

# 3. Imprimir el DataFrame completo usando la cuadrícula de tabulate
print(tabulate(df_total, headers='keys', tablefmt='grid', showindex=False))

# 4. Buena práctica: Restaurar los límites por defecto de Pandas para tu tranquilidad
pd.reset_option('display.max_rows')

DIMENSIONES DE ESTE ARCHIVO: (30, 30) (Filas, Columnas)

+----------+-------+---------+------------+-------------+-------------+-------------+---------------+---------------+---------------+----------------+----------------+----------------+----------------+------------------+------------------+------------------+------------------+----------------+----------------+----------------+------------------+------------------+------------------+------------+------------+------------+--------------+--------------+--------------+
| Sujeto   |   run |   tarea |   epoch_id |   PSD_mu_C3 |   PSD_mu_Cz |   PSD_mu_C4 |   PSD_beta_C3 |   PSD_beta_Cz |   PSD_beta_C4 |   CSP_mu_comp1 |   CSP_mu_comp2 |   CSP_mu_comp3 |   CSP_mu_comp4 |   CSP_beta_comp1 |   CSP_beta_comp2 |   CSP_beta_comp3 |   CSP_beta_comp4 |   MSC_mu_C3_C4 |   MSC_mu_C3_Cz |   MSC_mu_C4_Cz |   MSC_beta_C3_C4 |   MSC_beta_C3_Cz |   MSC_beta_C4_Cz |   PE_mu_C3 |   PE_mu_Cz |   PE_mu_C4 |   PE_beta_C3 |   PE_beta_Cz |   PE_beta_C4 |
+==

## Punto 2:  Rutina de procesamiento para la base de datos 

In [ ]:

# Ruta de la carpeta que contiene tus 30 archivos .set
carpeta_sujetos = 'sujetos'
patron_archivos = os.path.join(carpeta_sujetos, '*.set')
archivos_eeg = glob.glob(patron_archivos)

# Lista para acumular los DataFrames de cada archivo procesado
lista_dataframes_completos = []
claves_cruce = ['Sujeto', 'run', 'tarea', 'epoch_id']

print(f"Detectados {len(archivos_eeg)} archivos para procesamiento masivo.\n")

# 2. Bucle principal de procesamiento
for ruta_completa in archivos_eeg:
    nombre_archivo = os.path.basename(ruta_completa)
    
    try:
        # Procesamiento unificado del archivo actual
        (
            b0_er, b0_ei, b0_ed,
            b1_er, b1_ei, b1_ed,
            b2_er, b2_ei, b2_ed,
            sujeto, run
        ) = procesar_archivo_unificada(ruta_completa)
        
        # Validación de seguridad por si el archivo no contenía eventos válidos
        if b0_er is None:
            continue
            
        # Extracción individual de descriptores (3 tareas garantizadas)
        df_psd = construir_dataframe_psd(b0_er, b0_ei, b0_ed, sujeto, run)
        
        df_csp_mu, _   = extraer_features_csp(b1_er, b1_ei, b1_ed, sujeto, run, ritmo='mu')
        df_csp_beta, _ = extraer_features_csp(b1_er, b1_ei, b1_ed, sujeto, run, ritmo='beta')
        
        df_msc_mu   = extraer_features_msc(b0_er, b0_ei, b0_ed, sujeto, run, ritmo='mu')
        df_msc_beta = extraer_features_msc(b0_er, b0_ei, b0_ed, sujeto, run, ritmo='beta')
        
        df_pe_mu   = extraer_features_entropia(b2_er['mu'], b2_ei['mu'], b2_ed['mu'], sujeto, run, ritmo='mu')
        df_pe_beta = extraer_features_entropia(b2_er['beta'], b2_ei['beta'], b2_ed['beta'], sujeto, run, ritmo='beta')
        
        # Fusión horizontal estricta del archivo en proceso
        df_archivo = df_psd.copy()
        df_archivo = df_archivo.merge(df_csp_mu,   on=claves_cruce, how='inner')
        df_archivo = df_archivo.merge(df_csp_beta, on=claves_cruce, how='inner')
        df_archivo = df_archivo.merge(df_msc_mu,   on=claves_cruce, how='inner')
        df_archivo = df_archivo.merge(df_msc_beta, on=claves_cruce, how='inner')
        df_archivo = df_archivo.merge(df_pe_mu,   on=claves_cruce, how='inner')
        df_archivo = df_archivo.merge(df_pe_beta,  on=claves_cruce, how='inner')
        
        # Acumulación en la lista colectiva
        lista_dataframes_completos.append(df_archivo)
        
    except Exception as e:
        print(f"Error crítico al procesar el archivo {nombre_archivo}: {str(e)}")

# 3. Concatenación vertical para construir el Dataset Completo final
df_dataset_completo = pd.concat(lista_dataframes_completos, ignore_index=True)

print("Procesamiento masivo finalizado con éxito.")
print(f"Dimensiones totales del Dataset Completo: {df_dataset_completo.shape} (Filas, Columnas)\n")


# 4. Extracción y verificación del sujeto de prueba (sub-001, run 4)
# Filtramos de la matriz gigante únicamente el archivo que auditamos previamente
df_verificacion = df_dataset_completo[
    (df_dataset_completo['Sujeto'] == 'sub-001') & 
    (df_dataset_completo['run'] == '4')
]

print(f"Datos extraídos del procesamiento masivo para sub-001 (Run 4):")
print(f"Dimensiones del extracto: {df_verificacion.shape}\n")

# Impresión formateada con tabulate para tu auditoría visual de decimales
print(tabulate(df_verificacion, headers='keys', tablefmt='grid', showindex=False))

# Restaurar los límites de visualización de Pandas por seguridad de memoria
pd.reset_option('display.max_rows')

## Referencias 
[1] Blankertz, B., Tomioka, R., Lemm, S., Kawanabe, M., & Müller, K. R. (2007). Optimizing spatial filters for EEG-based brain–computer interfaces: a tutorial overview. IEEE Signal Processing Magazine, 24(1), 41-44. 

[2] Andrew, C., & Pfurtscheller, G. (1996). Event-related coherence as a tool for studying dynamic interaction of brain regions. Electroencephalography and Clinical Neurophysiology, 98(2), 144-148. 

[3] Bandt, C., & Pompe, B. (2002). Permutation entropy: a natural complexity measure for time series. Physical Review Letters, 88(17), 174102.